In [1]:
import functions as f
import pandas as pd
import numpy as np

train = pd.read_csv('data/train_fixed.csv')
X_train = train.drop(columns=['rainfall'])
y_train = train['rainfall']

test = pd.read_csv('data/prev/test_prev.csv')
#wybieramy pierwsze 13 kolumn
X_test = test.iloc[:, :13]

from sklearn.metrics import roc_auc_score
def evaluate_model(model, X_train, X_valid, y_train, y_valid):
    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_valid)[:,1]
    print(roc_auc_score(y_valid, y_pred))
    return roc_auc_score(y_valid, y_pred)

# Catboost
from catboost import CatBoostClassifier
y_train_r = y_train.values.reshape(-1)

categorical_features_indices = [i for i, col in enumerate(X_train.columns) if X_train[col].dtype == 'object']

model_catboost = CatBoostClassifier(
    iterations=600,
    learning_rate=0.05,
    depth=10,
    cat_features=categorical_features_indices,
    verbose=50
)
model_catboost.fit(X_train, y_train_r)
pred_catboost = model_catboost.predict_proba(X_test)[:, 1]

submission_cb = pd.DataFrame({
    'id': 2190+ X_test.index,
    'target': pred_catboost
})

submission_cb.to_csv('pred/pred_CatBoost1.csv', index=False)

#AdaBoost
from sklearn.ensemble import AdaBoostClassifier
y_train_ravel = y_train.to_numpy().ravel()

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
ada_model2 = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=2),
    n_estimators=600,
    learning_rate=0.05,
    #algorithm='SAMME',
    random_state=42
)

ada_model2.fit(X_train, y_train_ravel)
pred_ada = ada_model2.predict_proba(X_test)[:, 1]

submission_ada = pd.DataFrame({
    'id': 2190+ X_test.index,
    'target': pred_ada
})
submission_ada.to_csv('pred/pred_AdaBoost1.csv', index=False)


ada_model = AdaBoostClassifier(n_estimators=400, learning_rate=0.05)
ada_model.fit(X_train, y_train_ravel)
pred_ada = ada_model.predict_proba(X_test)[:, 1]

submission_ada = pd.DataFrame({
    'id': 2190+ X_test.index,
    'target': pred_ada
})
submission_ada.to_csv('pred/pred_AdaBoost2.csv', index=False)


0:	learn: 0.6492822	total: 253ms	remaining: 2m 31s
50:	learn: 0.2337854	total: 2.46s	remaining: 26.4s
100:	learn: 0.1731874	total: 3.94s	remaining: 19.5s
150:	learn: 0.1283424	total: 5.66s	remaining: 16.8s
200:	learn: 0.1001156	total: 7.29s	remaining: 14.5s
250:	learn: 0.0777673	total: 8.79s	remaining: 12.2s
300:	learn: 0.0607964	total: 10.4s	remaining: 10.3s
350:	learn: 0.0484318	total: 11.9s	remaining: 8.42s
400:	learn: 0.0401092	total: 13.4s	remaining: 6.64s
450:	learn: 0.0338941	total: 15.1s	remaining: 4.97s
500:	learn: 0.0290283	total: 16.7s	remaining: 3.31s
550:	learn: 0.0250991	total: 18.5s	remaining: 1.64s
599:	learn: 0.0222881	total: 20.2s	remaining: 0us


In [3]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=700,
    learning_rate=0.01,
    depth=6,
    l2_leaf_reg=1,
    cat_features=categorical_features_indices, 
    verbose=50,
    random_seed=42 
)

model.fit(X_train, y_train_r)
pred_catboost = model.predict_proba(X_test)[:, 1]

submission_cb = pd.DataFrame({
    'id': 2190+ X_test.index,
    'target': pred_catboost
})

submission_cb.to_csv('pred/pred_CatBoost3.csv', index=False)

0:	learn: 0.6850610	total: 3.79ms	remaining: 2.65s
50:	learn: 0.4431824	total: 151ms	remaining: 1.92s
100:	learn: 0.3613539	total: 285ms	remaining: 1.69s
150:	learn: 0.3264871	total: 445ms	remaining: 1.62s
200:	learn: 0.3062854	total: 602ms	remaining: 1.49s
250:	learn: 0.2931131	total: 743ms	remaining: 1.33s
300:	learn: 0.2831760	total: 906ms	remaining: 1.2s
350:	learn: 0.2741832	total: 1.05s	remaining: 1.04s
400:	learn: 0.2666735	total: 1.18s	remaining: 881ms
450:	learn: 0.2592199	total: 1.32s	remaining: 732ms
500:	learn: 0.2514845	total: 1.45s	remaining: 577ms
550:	learn: 0.2450445	total: 1.57s	remaining: 425ms
600:	learn: 0.2385461	total: 1.72s	remaining: 283ms
650:	learn: 0.2319312	total: 1.86s	remaining: 140ms
699:	learn: 0.2254188	total: 1.98s	remaining: 0us
